# 📦 Notebook 1 – Environment Setup

**Purpose:** Install all packages, mount Google Drive, check GPU, set global paths, and extract the dataset.

**Output:** A `config.json` file saved to `/content/` that all other notebooks read to know where the data lives.

**Run this notebook first, before any other notebook.**

---
Pipeline order:
```
01_setup  →  02_preprocess  →  03_yolo_train  →  04_pose_train  →  05_evaluate  →  06_visualize
```

| Step | Description |
|------|-------------|
| 1 | Mount Google Drive |
| 2 | Install all required packages |
| 3 | Verify GPU and print hardware info |
| 4 | Define and save global configuration |
| 5 | Clone the project repository from GitHub |
| 6 | Extract the LineMOD dataset zip |

## Cell 1 – Mount Google Drive

We mount Drive so we can:
- Read the dataset zip (`Linemod_preprocessed.zip`) stored on Drive
- Save trained model checkpoints back to Drive (survives session resets)

After running, click the authorization link that appears and paste the code.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted at /content/drive')

## Cell 2 – Install all required packages (single call)

Installing in one call is faster than multiple `!pip install` lines because pip resolves all dependencies at once instead of repeatedly.

| Package | Purpose |
|---------|----------|
| `torch / torchvision / torchaudio` | PyTorch with CUDA 11.8 support |
| `ultralytics` | YOLOv8 object detection |
| `open3d` | 3D point cloud, mesh loading, ICP |
| `opencv-python`, `Pillow` | Image I/O |
| `scikit-image` | Additional image processing |
| `scipy` | Quaternion / rotation math |
| `numpy`, `pandas`, `matplotlib` | Numerics, tables, plots |
| `tqdm` | Progress bars |
| `pyyaml` | Read `gt.yml` config files |
| `colorama` | Coloured terminal output in training logs |
| `numba` | JIT-compile radius-map kernel (10–100× faster) |

In [2]:
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q ultralytics open3d opencv-python Pillow scikit-image scipy \
              numpy pandas matplotlib tqdm pyyaml colorama numba

print('✅ All packages installed.')

## Cell 3 – Verify GPU and print hardware info

If `torch.cuda.is_available()` returns `False`:
- Go to **Runtime → Change runtime type → Hardware accelerator → GPU**

Best Colab GPUs for this project (from fastest to slowest):
`A100 (80 GB) > V100 (16 GB) > A100 (40 GB) > T4 (16 GB)`

In [3]:
import torch

print('─' * 55)
print(f'  PyTorch version : {torch.__version__}')
print(f'  CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  GPU model       : {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'  VRAM            : {vram:.1f} GB')
    print(f'  CUDA version    : {torch.version.cuda}')
    !nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader
else:
    print('  ⚠️  No GPU detected! Change runtime to GPU before continuing.')
print('─' * 55)

## Cell 4 – Define and save global configuration

All path constants and shared settings are defined here and saved to `/content/config.json`. Every other notebook loads this file at startup so they always use consistent paths — no copy-pasting paths between notebooks.

To change any path (e.g. if your zip is named differently), edit **only this cell**.

`ADD_THRESHOLDS`: per-class success thresholds for the ADD metric (metres). Source: standard LineMOD benchmark values.

In [ ]:
import os, json, numpy as np

DRIVE_ROOT = '/content/drive/MyDrive/Machine Learning Project/linemod/Linemod_preprocessed'

CONFIG = {
    'DRIVE_DATA_DIR'   : f'{DRIVE_ROOT}/data',    # source: per-class zips (01.zip ... 15.zip)
    'DRIVE_MODELS_DIR' : f'{DRIVE_ROOT}/models',  # source: obj_XX.ply models (folder or models.zip)
    'LINEMOD_ROOT'     : '/content/dataset/linemod',
    'DATA_DIR'         : '/content/dataset/linemod/Linemod_preprocessed/data',
    'MODELS_DIR_PLY'   : '/content/dataset/linemod/Linemod_preprocessed/models',
    'YOLO_DIR'         : '/content/dataset/linemod/Linemod_ready',
    'DRIVE_MODELS'     : '/content/drive/MyDrive/models',
    'REPO_DIR'         : '/content/6D-pose-estimation',
    'ALL_CLASSES'    : ['01','02','04','05','06','08','09','10','11','12','13','14','15'],
    'CLASS_NAMES'    : ['ape','benchvise','can','cat','driller',
                        'duck','eggbox','glue','holepuncher','iron','lamp','phone','camera'],
    'CAMERA_K'       : [[572.4114, 0.0, 325.2611],
                        [0.0, 573.57043, 242.04899],
                        [0.0, 0.0, 1.0]],
    'ADD_THRESHOLDS' : {
        '01':0.01421,'02':0.03309,'04':0.02223,'05':0.02842,
        '06':0.01859,'08':0.03188,'09':0.01557,'10':0.01974,
        '11':0.01931,'12':0.01961,'13':0.03172,'14':0.03166,'15':0.02543
    },
}

for d in [CONFIG['LINEMOD_ROOT'], CONFIG['DRIVE_MODELS']]:
    os.makedirs(d, exist_ok=True)

CONFIG_PATH = '/content/config.json'
with open(CONFIG_PATH, 'w') as f:
    json.dump(CONFIG, f, indent=2)

print(f'✅ Config saved to {CONFIG_PATH}')
print(f'   DATA_DIR    → {CONFIG["DATA_DIR"]}')
print(f'   DRIVE_MODELS → {CONFIG["DRIVE_MODELS"]}')

## Cell 5 – Clone the project repository from GitHub

The repo contains `src/model.py` and `src/dataset.py` which are shared modules imported by the training and evaluation notebooks.

If already cloned (e.g. from a previous run), the cell skips the clone step and does a `git pull` instead.

In [5]:
REPO_DIR = CONFIG['REPO_DIR']

if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/Sina-Ghiabi/6D-Pose-Estimation.git {REPO_DIR}
    print(f'✅ Repository cloned to {REPO_DIR}')
else:
    !cd {REPO_DIR} && git pull
    print(f'✅ Repository updated at {REPO_DIR}')

import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print('✅ sys.path updated — src modules are importable.')

## Cell 6 – Extract the LineMOD dataset to local Colab storage

Your Drive stores the dataset as **one zip per class** inside `Linemod_preprocessed/data/`
(`01.zip`, `02.zip`, ... `15.zip`) plus a `models/` folder (or `models.zip`) with the `.ply` meshes.

This cell reads each class zip from Drive and extracts it to **local Colab disk**
(`/content/dataset/linemod/...`) — same as the original project workflow. Extracting
onto the Drive mount itself doesn't actually avoid using local disk: Colab's DriveFS
still stages every write through local `/content` before syncing, so it hits the same
`No space left on device` error. Local disk is also just much faster for the thousands
of small file reads/writes that Notebook 2 (preprocessing) will do next.

**Expected result:**
```
/content/dataset/linemod/Linemod_preprocessed/data/01/rgb/*.png
/content/dataset/linemod/Linemod_preprocessed/data/01/depth/*.png
/content/dataset/linemod/Linemod_preprocessed/data/01/mask/*.png
/content/dataset/linemod/Linemod_preprocessed/data/01/gt.yml
/content/dataset/linemod/Linemod_preprocessed/models/obj_01.ply ... obj_15.ply
```

Already-extracted classes are skipped automatically, so this cell is safe to re-run.
If you hit a disk-space error partway through, run `!rm -rf /content/dataset` and
`Runtime → Disconnect and delete runtime` for a clean slate, then re-run from the top.

In [ ]:
import zipfile, shutil, uuid
from tqdm import tqdm

DATA_DIR = CONFIG['DATA_DIR']
os.makedirs(DATA_DIR, exist_ok=True)

MIN_FREE_GB = 5   # stop early instead of crashing mid-write if local disk gets low

def free_gb(path='/content'):
    return shutil.disk_usage(path).free / 1e9

def find_class_folder(root, class_id):
    """Walk the extracted tree and return the folder literally named `class_id`,
    no matter how deeply the zip nested it (some zips store bare rgb/depth/mask,
    some wrap in `<class_id>/`, some embed a full absolute path)."""
    for dirpath, dirnames, _ in os.walk(root):
        if os.path.basename(dirpath) == class_id:
            return dirpath
    return None

class_zips = sorted(f for f in os.listdir(CONFIG['DRIVE_DATA_DIR']) if f.endswith('.zip'))
missing = [z for z in class_zips
           if not os.path.isdir(os.path.join(DATA_DIR, os.path.splitext(z)[0]))]

if missing:
    print(f'📦 Extracting {len(missing)} class zip(s) from Drive → {DATA_DIR}')
    print(f'   Free space on /content right now: {free_gb():.1f} GB')
    for zname in tqdm(missing, desc='Extracting classes', unit='class'):
        if free_gb() < MIN_FREE_GB:
            print(f'\n🛑 Stopping: only {free_gb():.1f} GB free on /content (< {MIN_FREE_GB} GB threshold).')
            print('   Free up space (e.g. Runtime → Disconnect and delete runtime) and re-run this cell.')
            break

        class_id  = os.path.splitext(zname)[0]
        zpath     = os.path.join(CONFIG['DRIVE_DATA_DIR'], zname)
        final_dir = os.path.join(DATA_DIR, class_id)
        tmp_dir   = os.path.join(DATA_DIR, f'.tmp_extract_{class_id}_{uuid.uuid4().hex[:8]}')
        os.makedirs(tmp_dir, exist_ok=True)
        try:
            with zipfile.ZipFile(zpath, 'r') as zf:
                zf.extractall(tmp_dir)
            # If the zip's own root *is* the class contents (rgb/, depth/, ... directly)
            # rather than a folder named `class_id`, treat tmp_dir itself as the source.
            src = find_class_folder(tmp_dir, class_id)
            if src is None and any(f in os.listdir(tmp_dir) for f in ('rgb', 'depth', 'mask', 'gt.yml')):
                src = tmp_dir
            if src is None:
                print(f'   ⚠️  Could not locate class-{class_id} contents inside {zname} — inspect it manually.')
                continue
            if src == tmp_dir:
                os.rename(tmp_dir, final_dir)
            else:
                shutil.move(src, final_dir)
        finally:
            if os.path.isdir(tmp_dir):
                shutil.rmtree(tmp_dir, ignore_errors=True)
    print('✅ Class data extracted.')
else:
    print(f'⏩ All classes already extracted at {DATA_DIR}.')

# --- 3D models (.ply) ---
MODELS_DIR_PLY = CONFIG['MODELS_DIR_PLY']
if not os.path.isdir(MODELS_DIR_PLY):
    if os.path.isdir(CONFIG['DRIVE_MODELS_DIR']):
        print(f'📦 Copying 3D models → {MODELS_DIR_PLY}')
        shutil.copytree(CONFIG['DRIVE_MODELS_DIR'], MODELS_DIR_PLY)
        print('✅ Models copied.')
    else:
        models_zip = CONFIG['DRIVE_MODELS_DIR'] + '.zip'
        if os.path.isfile(models_zip):
            print(f'📦 Extracting models zip → {MODELS_DIR_PLY}')
            with zipfile.ZipFile(models_zip, 'r') as zf:
                zf.extractall(os.path.dirname(MODELS_DIR_PLY))
            print('✅ Models extracted.')
        else:
            print(f'⚠️  No models found at {CONFIG["DRIVE_MODELS_DIR"]} (or .zip) — check the path.')
else:
    print(f'⏩ Models already at {MODELS_DIR_PLY}.')

found = sorted(d for d in os.listdir(DATA_DIR)
               if os.path.isdir(os.path.join(DATA_DIR, d)))
print(f'\nClass folders found: {found}')

---
## ✅ Setup Complete

- ✅ Google Drive mounted
- ✅ All packages installed
- ✅ GPU verified
- ✅ Config saved to `/content/config.json`
- ✅ Repository cloned
- ✅ Dataset extracted

**Next step → Open and run `02_preprocess.ipynb`**